In [ ]:
# Scenario Analysis and Comparison with ws3

This notebook demonstrates how to perform scenario analysis and compare different management strategies using `ws3`. You'll learn how to formulate multiple optimization problems with different objectives and constraints, solve them, and compare the results.

> **Prerequisites**: Completion of the basic ws3 workflow (see `070_ws3_quickstart_complete_workflow.ipynb`)

## What You'll Learn

- How to formulate multiple optimization scenarios
- How to modify model parameters between scenarios
- How to solve and compare scenarios
- How to visualize trade-offs between objectives
- How to perform sensitivity analysis

## Scenario Types

We'll explore several common scenario types:

1. **Even-Flow vs. Maximization**: Compare even-flow harvesting with maximum harvest volume
2. **Different Time Horizons**: Analyze how planning horizon affects outcomes
3. **Carbon Considerations**: Include carbon objectives
4. **Spatial Constraints**: Add adjacency and contiguous area constraints
5. **Sensitivity Analysis**: Test robustness to parameter changes

## Step 1: Environment Setup

```python
%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
import ws3.core
from util import run_scenario

# Set up figure styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
```

## Step 2: Load and Prepare Data

```python
# Model parameters
base_year = 2020
period_length = 10
max_age = 1000
tvy_name = "totvol"

# Load data
stands = gpd.read_file("data/shp/tsa24_clipped.shp/stands.shp")
au_table = pd.read_csv("data/au_table.csv").set_index("au_id")
curve_table = pd.read_csv("data/curve_table.csv")
curve_points_table = pd.read_csv("data/curve_points_table.csv").set_index("curve_id")

# Prepare THLB
au_table["thlb"] = au_table.apply(
    lambda row: 0 if row.unmanaged_curve_id == row.managed_curve_id else 1, 
    axis=1
)
stands["theme1"] = stands.apply(
    lambda row: au_table.loc[row.theme2].thlb, 
    axis=1
)
stands["theme4"] = stands.curve1

print(f"Data loaded: {len(stands)} stands, {len(au_table)} AUs")
```

## Step 3: Define Scenario Function

Let's create a reusable function to run scenarios with different parameters.

```python
def run_optimization_scenario(fm, scenario_name, objective_type="even_flow", 
                              horizon=10, workers=1):
    """
    Run an optimization scenario with specified parameters.
    
    Parameters:
    -----------
    fm : ForestModel
        The forest model to optimize
    scenario_name : str
        Name for this scenario
    objective_type : str
        Type of objective: 'even_flow', 'maximize', 'minimize'
    horizon : int
        Number of periods
    workers : int
        Number of parallel workers
    
    Returns:
    --------
    problem : OptimizationProblem
        The formulated problem
    solution : Solution
        The solved solution
    """
    # Formulate problem
    problem = run_scenario(
        fm, 
        scenario_name=scenario_name, 
        print_df=False, 
        workers=workers
    )
    
    # Modify objective based on type
    if objective_type == "even_flow":
        # Even-flow maximizes the minimum period harvest
        problem.objective = "max_min"
    elif objective_type == "maximize":
        # Maximize total harvest volume
        problem.objective = "max_sum"
    elif objective_type == "minimize":
        # Minimize total harvest (conservation)
        problem.objective = "min_sum"
    
    # Solve
    solution = problem.solve(solver="gurobi")
    
    return problem, solution
```

## Step 4: Run Multiple Scenarios

Let's run several scenarios to compare different approaches.

```python
# Create base model
fm = ws3.forest.ForestModel(
    model_name="scenario_comparison",
    model_path="data/woodstock_model_files_tsa24_clipped",
    base_year=base_year,
    horizon=10,
    period_length=period_length,
    max_age=max_age
)

fm.import_landscape_section()
fm.import_areas_section(convert_periods_to_years=period_length)
fm.import_yields_section(convert_periods_to_years=period_length)
fm.import_actions_section(convert_periods_to_years=period_length)
fm.import_transitions_section(convert_periods_to_years=period_length)
fm.initialize_areas()
fm.add_null_action()
fm.reset_actions()
fm.actions["harvest"].is_harvest = True

# Define scenarios
scenarios = [
    ("even_flow_10yr", "even_flow", 10),
    ("maximize_10yr", "maximize", 10),
    ("even_flow_20yr", "even_flow", 20),
    ("maximize_20yr", "maximize", 20),
    ("minimize_10yr", "minimize", 10),
]

# Run all scenarios
results = {}
for name, obj_type, horizon in scenarios:
    print(f"Running scenario: {name}...")
    problem, solution = run_optimization_scenario(
        fm, name, obj_type, horizon, workers=1
    )
    results[name] = {
        'problem': problem,
        'solution': solution,
        'objective_type': obj_type,
        'horizon': horizon
    }
    print(f"  ✓ Completed (objective value: {solution.objective_value:.2f})")

print(f"\nAll {len(results)} scenarios completed!")
```

## Step 5: Extract Results

```python
def extract_period_volumes(solution):
    """Extract harvest volumes by period from solution."""
    harvest_volumes = solution.get_variable_values("x")
    period_volumes = {}
    for au_id, action, period, volume in harvest_volumes:
        if action == "harvest":
            period_volumes[period] = period_volumes.get(period, 0) + volume
    return period_volumes

# Extract volumes for all scenarios
scenario_volumes = {}
for name, data in results.items():
    scenario_volumes[name] = extract_period_volumes(data['solution'])

# Create summary table
summary_data = []
for name, data in results.items():
    volumes = scenario_volumes[name]
    summary_data.append({
        'Scenario': name,
        'Objective': data['objective_type'],
        'Horizon': data['horizon'],
        'Total Volume': sum(volumes.values()),
        'Min Period': min(volumes.values()) if volumes else 0,
        'Max Period': max(volumes.values()) if volumes else 0,
        'Std Dev': np.std(list(volumes.values())) if volumes else 0,
        'Objective Value': data['solution'].objective_value
    })

summary_df = pd.DataFrame(summary_data)
summary_df
```

## Step 6: Visualize Scenario Comparisons

```python
# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Total harvest volume by scenario
ax = axes[0, 0]
ax.barh(range(len(summary_df)), summary_df['Total Volume'])
ax.set_yticks(range(len(summary_df)))
ax.set_yticklabels(summary_df['Scenario'])
ax.set_xlabel('Total Harvest Volume')
ax.set_title('Total Harvest Volume by Scenario')

# Plot 2: Even-flow consistency (standard deviation)
ax = axes[0, 1]
ax.barh(range(len(summary_df)), summary_df['Std Dev'])
ax.set_yticks(range(len(summary_df)))
ax.set_yticklabels(summary_df['Scenario'])
ax.set_xlabel('Standard Deviation (Even-Flow Measure)')
ax.set_title('Even-Flow Consistency by Scenario')

# Plot 3: Harvest volume by period for even-flow scenarios
ax = axes[1, 0]
for name in ['even_flow_10yr', 'even_flow_20yr']:
    volumes = scenario_volumes[name]
    periods = sorted(volumes.keys())
    ax.plot(periods, [volumes[p] for p in periods], marker='o', label=name)
ax.set_xlabel('Period')
ax.set_ylabel('Harvest Volume')
ax.set_title('Even-Flow Scenarios: Volume by Period')
ax.legend()

# Plot 4: Comparison of objectives (10-year horizon)
ax = axes[1, 1]
scenarios_10yr = ['even_flow_10yr', 'maximize_10yr', 'minimize_10yr']
volumes_dict = {name: scenario_volumes[name] for name in scenarios_10yr}
periods = sorted(volumes_dict[scenarios_10yr[0]].keys())

x = np.arange(len(periods))
width = 0.25

for i, name in enumerate(scenarios_10yr):
    vols = [volumes_dict[name].get(p, 0) for p in periods]
    ax.bar(x + i*width, vols, width, label=name)

ax.set_xlabel('Period')
ax.set_ylabel('Harvest Volume')
ax.set_title('10-Year Scenarios: Volume by Period')
ax.set_xticks(x + width)
ax.set_xticklabels([f'P{p+1}' for p in periods])
ax.legend()

plt.tight_layout()
plt.show()
```

## Step 7: Sensitivity Analysis

Let's perform sensitivity analysis on the planning horizon.

```python
# Test different horizon lengths
horizons = [5, 10, 15, 20, 25, 30]
horizon_results = []

print("Running sensitivity analysis on planning horizon...")
for h in horizons:
    print(f"  Testing horizon: {h} years...")
    problem, solution = run_optimization_scenario(
        fm, f"sensitivity_{h}", "even_flow", h, workers=1
    )
    volumes = extract_period_volumes(solution)
    horizon_results.append({
        'Horizon': h,
        'Objective Value': solution.objective_value,
        'Total Volume': sum(volumes.values()),
        'Min Period': min(volumes.values()) if volumes else 0,
        'Avg Period': np.mean(list(volumes.values())) if volumes else 0,
        'Std Dev': np.std(list(volumes.values())) if volumes else 0
    })

horizon_df = pd.DataFrame(horizon_results)
horizon_df
```

## Step 8: Visualize Sensitivity Analysis

```python
# Sensitivity analysis visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Objective value vs horizon
ax = axes[0]
ax.plot(horizon_df['Horizon'], horizon_df['Objective Value'], marker='o', linewidth=2)
ax.set_xlabel('Planning Horizon (years)')
ax.set_ylabel('Objective Value')
ax.set_title('Objective Value vs Planning Horizon')
ax.grid(True, alpha=0.3)

# Plot 2: Even-flow consistency vs horizon
ax = axes[1]
ax.plot(horizon_df['Horizon'], horizon_df['Std Dev'], marker='s', linewidth=2, color='orange')
ax.set_xlabel('Planning Horizon (years)')
ax.set_ylabel('Standard Deviation')
ax.set_title('Even-Flow Consistency vs Planning Horizon')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
```

## Step 9: Identify Best Scenario

```python
# Define criteria for "best" scenario
# We want: high total volume, low standard deviation (even-flow)

# Normalize metrics (0-1 scale, higher is better)
summary_df['Volume_Norm'] = (summary_df['Total Volume'] - summary_df['Total Volume'].min()) / \
                           (summary_df['Total Volume'].max() - summary_df['Total Volume'].min())
summary_df['EvenFlow_Norm'] = 1 - (summary_df['Std Dev'] - summary_df['Std Dev'].min()) / \
                              (summary_df['Std Dev'].max() - summary_df['Std Dev'].min())

# Weighted score (equal weight for now)
summary_df['Composite_Score'] = 0.5 * summary_df['Volume_Norm'] + 0.5 * summary_df['EvenFlow_Norm']

# Rank scenarios
summary_df = summary_df.sort_values('Composite_Score', ascending=False)
summary_df[['Scenario', 'Total Volume', 'Std Dev', 'Composite_Score']]
```

## Step 10: Export Results

```python
# Export scenario summary
summary_df.to_csv("scenario_comparison_summary.csv", index=False)

# Export individual scenario results
for name, data in results.items():
    volumes = scenario_volumes[name]
    df = pd.DataFrame([
        {'Period': p, 'Volume': v} 
        for p, v in volumes.items()
    ])
    df.to_csv(f"scenario_{name}_volumes.csv", index=False)

print("Results exported:")
print("  - scenario_comparison_summary.csv")
print("  - scenario_*_volumes.csv files")

# Show best scenario
best_scenario = summary_df.iloc[0]
print(f"\nBest Scenario: {best_scenario['Scenario']}")
print(f"  Total Volume: {best_scenario['Total Volume']:.2f}")
print(f"  Even-Flow Score: {best_scenario['Composite_Score']:.3f}")
```

## Step 11: Advanced Analysis - Trade-off Curves

Let's explore the trade-off between total volume and even-flow.

```python
# Create trade-off analysis
trade_off_data = []
for name, data in results.items():
    volumes = scenario_volumes[name]
    trade_off_data.append({
        'Scenario': name,
        'Total Volume': sum(volumes.values()),
        'EvenFlow_Quality': 1.0 / (1.0 + np.std(list(volumes.values())))
    })

trade_off_df = pd.DataFrame(trade_off_data)

# Plot trade-off curve
plt.figure(figsize=(10, 6))
plt.scatter(trade_off_df['Total Volume'], trade_off_df['EvenFlow_Quality'], 
            s=100, alpha=0.7, edgecolors='black')

# Add labels
for i, row in trade_off_df.iterrows():
    plt.annotate(row['Scenario'], 
                 (row['Total Volume'], row['EvenFlow_Quality']),
                 xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('Total Harvest Volume')
plt.ylabel('Even-Flow Quality (1 / (1 + StdDev))')
plt.title('Trade-off: Total Volume vs Even-Flow Quality')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

## Summary

In this notebook, you learned how to:

1. ✓ Formulate multiple optimization scenarios with different objectives
2. ✓ Run scenarios systematically and collect results
3. ✓ Compare scenarios using quantitative metrics
4. ✓ Visualize trade-offs between competing objectives
5. ✓ Perform sensitivity analysis on planning parameters
6. ✓ Identify optimal scenarios based on composite criteria
7. ✓ Export results for further analysis

## Key Insights

- **Even-flow vs. Maximization**: Even-flow scenarios provide more stable harvest levels but may sacrifice total volume
- **Planning Horizon**: Longer horizons can improve even-flow but may reduce near-term harvest opportunities
- **Trade-offs**: There are inherent trade-offs between different objectives that require careful consideration

## Next Steps

- Explore multi-objective optimization to find Pareto-optimal solutions
- Add spatial constraints (adjacency, contiguous area)
- Incorporate carbon accounting with libCBM
- Perform Monte Carlo simulation for uncertainty analysis

## Troubleshooting

**Common Issues:**

1. **Long solve times**: Reduce horizon or use parallel solving
   ```python
   solution = problem.solve(solver="gurobi", workers=4)
   ```

2. **Infeasible scenarios**: Check constraints and data validity
   - Verify yield curves are defined for all areas
   - Ensure harvest actions are properly configured
   - Check age and area constraints

3. **Memory errors**: Process scenarios sequentially
   ```python
   # Run one scenario at a time
   for name in scenario_names:
       run_single_scenario(name)
   ```

## References

- [ws3 Documentation](https://ws3.readthedocs.io)
- [Optimization Theory](https://en.wikipedia.org/wiki/Mathematical_optimization)
- [Multi-Objective Optimization](https://en.wikipedia.org/wiki/Mathematical_optimization#Multi-objective_optimization)
- [Sensitivity Analysis](https://en.wikipedia.org/wiki/Sensitivity_analysis)